# Temporal GNN — Novelty Evolution Modeling

**Member 3 — Temporal Modeling & Evolution Lead**

This notebook implements:
1. **KG(t) Snapshots** — yearly cumulative knowledge-graph slices
2. **Temporal GNN (TGAT-style)** — time-encoding + graph attention for novelty scoring
3. **Link Prediction** — predict future knowledge edges across yearly windows
4. **Novelty Time-Series Modeling** — track how novelty evolves year by year
5. **Emerging Concept Detection** — identify entities gaining traction over time
6. **Incremental vs Disruptive Classification** — characterise contribution type
7. **Outputs** — save all results to `outputs/final/`

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import defaultdict
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import MinMaxScaler
import warnings, os

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


In [2]:
# ── Paths ─────────────────────────────────────────────────────
BASE       = '../../outputs'
EDGES_PATH  = f'{BASE}/final/knowledge_edges.csv'
PAPERS_PATH = f'{BASE}/final/paper_nodes.csv'
MATRIX_PATH = f'{BASE}/final/novelty_feature_matrix_with_score.csv'
OUT_DIR     = f'{BASE}/final'

os.makedirs(OUT_DIR, exist_ok=True)

## 1 — Load Data & Build KG(t) Snapshots

In [3]:
try:
    edges  = pd.read_csv(EDGES_PATH)
    papers = pd.read_csv(PAPERS_PATH)
    matrix = pd.read_csv(MATRIX_PATH)
except (FileNotFoundError, pd.errors.ParserError) as e:
    raise RuntimeError(f'Failed to load data: {e}. Check that outputs/final/ exists.') from e


# Clean year
edges  = edges.dropna(subset=['year'])
edges['year'] = edges['year'].astype(int)
papers = papers.dropna(subset=['year'])
papers['year'] = papers['year'].astype(int)

# Attach split to edges
paper_split = dict(zip(papers['node_id'], papers['split']))
paper_year  = dict(zip(papers['node_id'], papers['year']))
edges['split'] = edges['source'].map(paper_split)

print(f"Total edges : {len(edges):,}")
print(f"Year range  : {edges.year.min()} – {edges.year.max()}")
print(f"SKG papers  : {(papers.split=='SKG').sum():,}")
print(f"NOVEL papers: {(papers.split=='NOVEL').sum():,}")

Total edges : 577,024
Year range  : 2010 – 2025
SKG papers  : 3,166
NOVEL papers: 854


In [4]:
# ── KG(t) = all knowledge until (and including) year t ────────
def build_snapshot(edges_df: pd.DataFrame, t: int) -> pd.DataFrame:
    """Return all edges with year <= t (cumulative historical KG)."""
    return edges_df[edges_df['year'] <= t].copy()

ALL_YEARS = sorted(edges['year'].unique())

# Edge-addition dynamics — cumulative growth curve
snapshot_stats = []
seen_triples   = set()
seen_entities  = set()

for yr in ALL_YEARS:
    yr_edges = edges[edges['year'] == yr]
    new_triples  = set(zip(yr_edges.source, yr_edges.predicate, yr_edges.target))
    new_entities = set(yr_edges['source']) | set(yr_edges['target'])
    novel_triples = new_triples - seen_triples
    seen_triples  |= new_triples
    seen_entities |= new_entities
    snapshot_stats.append({
        'year'              : yr,
        'edges_added'       : len(yr_edges),
        'novel_triples'     : len(novel_triples),
        'cumulative_edges'  : len(seen_triples),
        'cumulative_entities': len(seen_entities),
    })

snapshot_df = pd.DataFrame(snapshot_stats)
print(snapshot_df.to_string(index=False))

 year  edges_added  novel_triples  cumulative_edges  cumulative_entities
 2010         4242           4242              4242                 1953
 2011         4421           4421              8663                 3817
 2012         5648           5648             14311                 6072
 2013        10869          10869             25180                10108
 2014         7525           7525             32705                12820
 2015        10824          10824             43529                16636
 2016        14643          14643             58172                21508
 2017        22937          22937             81109                28692
 2018        34647          34647            115756                39210
 2019        54148          54148            169904                54881
 2020        64956          64956            234860                73504
 2021        51435          51435            286295                87455
 2022       134200         134200            420495

## 2 — Temporal GNN (TGAT-Style) for Novelty Scoring

Architecture:
- **Time encoding** — Bochner-inspired harmonic time embedding (from TGAT)
- **Graph attention** — single-layer multi-head attention aggregating neighbourhood features
- **Novelty head** — MLP that scores how "surprising" a paper's edge-set is given KG(t-1)

In [5]:
class TimeEncoding(nn.Module):
    """Bochner time encoding (TGAT, Xu et al. 2020)."""
    def __init__(self, d_time: int):
        super().__init__()
        self.d_time = d_time
        self.w = nn.Linear(1, d_time)
        nn.init.normal_(self.w.weight, std=0.1)
        nn.init.zeros_(self.w.bias)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t : (B,) float
        t = t.unsqueeze(-1).float()           # (B, 1)
        phi = self.w(t)                        # (B, d_time)
        return torch.cos(phi)                  # (B, d_time)


class TemporalNoveltyGNN(nn.Module):
    """
    Lightweight TGAT-inspired model.
    Input features per paper: [structural_novelty, semantic_knn, triple_count_norm, citation_norm]
    Time encoding is concatenated to features before a 2-layer MLP novelty head.
    """
    def __init__(self, in_dim: int, d_time: int, hidden: int):
        super().__init__()
        self.time_enc = TimeEncoding(d_time)
        self.fc1 = nn.Linear(in_dim + d_time, hidden)
        self.fc2 = nn.Linear(hidden, hidden // 2)
        self.head = nn.Linear(hidden // 2, 1)
        self.drop = nn.Dropout(0.2)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        te = self.time_enc(t)                 # (B, d_time)
        h  = torch.cat([x, te], dim=-1)       # (B, in_dim+d_time)
        h  = F.relu(self.fc1(h))
        h  = self.drop(h)
        h  = F.relu(self.fc2(h))
        return torch.sigmoid(self.head(h)).squeeze(-1)   # (B,)

print("TGNN model classes defined.")

TGNN model classes defined.


In [6]:
# ── Prepare feature matrix ─────────────────────────────────────
matrix = matrix.merge(
    papers[['node_id','split','domain','title']],
    left_on='paper_id', right_on='node_id', how='left'
)

FEAT_COLS = ['structural_novelty', 'semantic_knn', 'struct_norm', 'semantic_norm', 'citation_norm']
matrix[FEAT_COLS] = matrix[FEAT_COLS].fillna(matrix[FEAT_COLS].median())

# Normalise year to [0, 1] for time encoding
y_min, y_max = matrix['year'].min(), matrix['year'].max()
matrix['year_norm'] = (matrix['year'] - y_min) / max(y_max - y_min, 1)

# Binary label: 1 = NOVEL paper
matrix['label'] = (matrix['split'] == 'NOVEL').astype(float)

# Train on SKG; evaluate on NOVEL+SKG
train_df = matrix[matrix['split'] == 'SKG'].copy()
eval_df  = matrix[matrix['split'].isin(['SKG','NOVEL'])].copy()

print(f"Train samples: {len(train_df):,}   Eval samples: {len(eval_df):,}")

Train samples: 2,916   Eval samples: 3,678


In [7]:
IN_DIM  = len(FEAT_COLS)
D_TIME  = 16
HIDDEN  = 64
LR      = 1e-3
EPOCHS  = 40
BS      = 256

X_train = torch.tensor(train_df[FEAT_COLS].values, dtype=torch.float32).to(DEVICE)
T_train = torch.tensor(train_df['year_norm'].values, dtype=torch.float32).to(DEVICE)
y_train = torch.tensor(train_df['label'].values, dtype=torch.float32).to(DEVICE)

X_eval  = torch.tensor(eval_df[FEAT_COLS].values, dtype=torch.float32).to(DEVICE)
T_eval  = torch.tensor(eval_df['year_norm'].values, dtype=torch.float32).to(DEVICE)
y_eval  = torch.tensor(eval_df['label'].values, dtype=torch.float32).to(DEVICE)

model = TemporalNoveltyGNN(IN_DIM, D_TIME, HIDDEN).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCELoss()

n = len(X_train)
best_auc = 0.0
for ep in range(1, EPOCHS + 1):
    model.train()
    idx = torch.randperm(n)
    ep_loss = 0.0
    for start in range(0, n, BS):
        batch = idx[start:start+BS]
        pred  = model(X_train[batch], T_train[batch])
        loss  = loss_fn(pred, y_train[batch])
        opt.zero_grad(); loss.backward(); opt.step()
        ep_loss += loss.item()

    if ep % 10 == 0:
        model.eval()
        with torch.no_grad():
            scores = model(X_eval, T_eval).cpu().numpy()
        auc = roc_auc_score(y_eval.cpu().numpy(), scores)
        ap  = average_precision_score(y_eval.cpu().numpy(), scores)
        if auc > best_auc:
            best_auc = auc
        print(f"Epoch {ep:3d} | loss={ep_loss:.4f} | AUC={auc:.4f} | AP={ap:.4f}")

print(f"\nBest eval AUC: {best_auc:.4f}")

Epoch  10 | loss=0.0084 | AUC=0.6661 | AP=0.3237
Epoch  20 | loss=0.0015 | AUC=0.6649 | AP=0.3232
Epoch  30 | loss=0.0006 | AUC=0.6642 | AP=0.3230
Epoch  40 | loss=0.0003 | AUC=0.6638 | AP=0.3228

Best eval AUC: 0.6661


In [8]:
# ── Score all papers and save ──────────────────────────────────
model.eval()
X_all = torch.tensor(matrix[FEAT_COLS].values, dtype=torch.float32).to(DEVICE)
T_all = torch.tensor(matrix['year_norm'].values, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    tgnn_scores = model(X_all, T_all).cpu().numpy()

matrix['tgnn_novelty'] = tgnn_scores

# Save TGNN scores
tgnn_out = matrix[['paper_id','year','split','domain','tgnn_novelty']].copy()
tgnn_out.to_csv(f'{OUT_DIR}/tgnn_novelty_scores.csv', index=False)
print(f"Saved tgnn_novelty_scores.csv  ({len(tgnn_out)} rows)")

# Quick distribution check
for sp in ['SKG','NOVEL']:
    sub = matrix[matrix['split']==sp]['tgnn_novelty']
    print(f"  {sp}: mean={sub.mean():.4f}  std={sub.std():.4f}  n={len(sub)}")

Saved tgnn_novelty_scores.csv  (4242 rows)
  SKG: mean=0.0000  std=0.0000  n=2916
  NOVEL: mean=0.0000  std=0.0000  n=762


## 3 — Link Prediction Across Temporal Windows

For each consecutive year pair (t, t+1):  
- **Positive edges**: edges that first appear in year t+1  
- **Negative edges**: randomly sampled entity pairs not in KG(t+1)  
- **Score**: cosine similarity of TransE-style entity co-occurrence vectors

In [10]:
# ── Build entity co-occurrence vectors from KG(t) ─────────────
# For each entity, count how many unique entities it connects to per year

skg_edges = edges[edges['split'] == 'SKG'].copy()

EVAL_WINDOWS = [(y, y+1) for y in range(2015, 2018)]   # 3 test windows on SKG

from scipy.sparse import csr_matrix

def build_entity_cooccurrence(train_edges: pd.DataFrame, dim: int = 32) -> dict:
    """Build a co-occurrence embedding per entity using a sparse random projection."""
    all_ents = pd.unique(pd.concat([train_edges['source'], train_edges['target']]))
    ent2idx  = {e: i for i, e in enumerate(all_ents)}
    n = len(all_ents)
    src_idx = train_edges['source'].map(ent2idx).dropna().astype(int).values
    tgt_idx = train_edges['target'].map(ent2idx).dropna().astype(int).values
    sz = min(len(src_idx), len(tgt_idx))
    si, ti = src_idx[:sz], tgt_idx[:sz]
    data = np.ones(2*sz, dtype=np.float32)
    rows = np.concatenate([si, ti])
    cols = np.concatenate([ti, si])
    mat  = csr_matrix((data, (rows, cols)), shape=(n, n))
    # Row-normalise
    row_sums = np.asarray(mat.sum(axis=1)).flatten() + 1e-9
    # Random projection: mat (n,n) @ proj (n,dim)
    # Use a sparse random projection: draw a dense (n, dim) is OK for dim=32
    rng_loc = np.random.default_rng(SEED)
    proj = rng_loc.standard_normal((n, dim)).astype(np.float32) / np.sqrt(dim)
    emb  = mat.dot(proj) / row_sums[:, None]   # (n, dim)
    return {e: emb[ent2idx[e]] for e in all_ents}


def score_link(e1: str, e2: str, emb: dict) -> float:
    """Cosine similarity of two entity embeddings (0 if unknown)."""
    v1, v2 = emb.get(e1), emb.get(e2)
    if v1 is None or v2 is None:
        return 0.0
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9))


rng = np.random.default_rng(SEED)
lp_results = []
# SAMPLE_POS=200 balances statistical power (≥200 pairs gives stable AUC estimates)
# and runtime (avoids full cartesian product over 10k+ entities per window).
SAMPLE_POS  = 200

for t, t1 in EVAL_WINDOWS:
    train_e = skg_edges[skg_edges['year'] <= t]
    test_e  = skg_edges[skg_edges['year'] == t1]

    if len(test_e) == 0:
        continue

    emb = build_entity_cooccurrence(train_e, dim=32)

    pos_pairs = list(zip(test_e['source'], test_e['target']))
    rng.shuffle(pos_pairs)
    pos_pairs = pos_pairs[:SAMPLE_POS]
    all_ents  = list(emb.keys())

    # Sample equal number of negatives
    neg_pairs = []
    pos_set   = set(pos_pairs)
    attempts  = 0
    while len(neg_pairs) < len(pos_pairs) and attempts < 10000:
        a, b = rng.choice(all_ents, 2, replace=False)
        if (a, b) not in pos_set:
            neg_pairs.append((a, b))
        attempts += 1

    scores = ([score_link(a, b, emb) for a, b in pos_pairs] +
              [score_link(a, b, emb) for a, b in neg_pairs])
    labels = [1]*len(pos_pairs) + [0]*len(neg_pairs)

    auc = roc_auc_score(labels, scores)
    ap  = average_precision_score(labels, scores)
    lp_results.append({'window': f'{t}→{t1}', 'pos': len(pos_pairs), 'AUC': auc, 'AP': ap})
    print(f"  {t}→{t1}  pos={len(pos_pairs):,}  AUC={auc:.4f}  AP={ap:.4f}")

lp_df = pd.DataFrame(lp_results)
lp_df.to_csv(f'{OUT_DIR}/link_prediction_results.csv', index=False)
print(f"\nMean AUC: {lp_df['AUC'].mean():.4f}  |  Mean AP: {lp_df['AP'].mean():.4f}")
print("Saved link_prediction_results.csv")

  2015→2016  pos=200  AUC=0.4850  AP=0.6601
  2016→2017  pos=200  AUC=0.5350  AP=0.6826
  2017→2018  pos=200  AUC=0.4400  AP=0.6410

Mean AUC: 0.4867  |  Mean AP: 0.6612
Saved link_prediction_results.csv


## 4 — Novelty Time-Series Modeling

Track mean novelty score per year and fit a trend model to characterise whether novelty is increasing or decreasing over time.

In [11]:
# ── Year-level novelty statistics ─────────────────────────────
yr_stats = (
    matrix[matrix['split'].isin(['SKG','NOVEL'])]
    .groupby(['year','split'])['final_novelty_score']
    .agg(mean='mean', std='std', count='count')
    .reset_index()
)

# ── Novelty vs year trend (Spearman correlation) ───────────────
for sp in ['SKG','NOVEL']:
    sub = yr_stats[yr_stats['split']==sp].dropna()
    if len(sub) >= 3:
        rho, p = spearmanr(sub['year'], sub['mean'])
        trend = 'increasing' if rho > 0 else 'decreasing'
        print(f"{sp}: Spearman ρ={rho:.3f}  p={p:.4f}  → novelty is {trend} over time")

print()
print(yr_stats.to_string(index=False))

SKG: Spearman ρ=0.421  p=0.1177  → novelty is increasing over time
NOVEL: Spearman ρ=0.486  p=0.3287  → novelty is increasing over time

 year split     mean      std  count
 2010   SKG 0.253351 0.042471     59
 2011   SKG 0.341464 0.063234     64
 2012   SKG 0.343722 0.083119     73
 2013   SKG 0.330436 0.055536    122
 2014   SKG 0.329947 0.058239     89
 2015   SKG 0.398877 0.112795    116
 2016   SKG 0.334098 0.058651    129
 2017   SKG 0.333093 0.066231    221
 2018   SKG 0.338282 0.066304    290
 2019   SKG 0.381237 0.096393    436
 2020 NOVEL 0.340144 0.054625      4
 2020   SKG 0.322409 0.049105    500
 2021 NOVEL 0.372922 0.095917    403
 2022 NOVEL 0.410748 0.055305    106
 2022   SKG 0.389108 0.055031    326
 2023 NOVEL 0.367370 0.035141    218
 2023   SKG 0.366812 0.042951    418
 2024 NOVEL 0.356942 0.029558     24
 2024   SKG 0.352313 0.034162     69
 2025 NOVEL 0.426166 0.133111      7
 2025   SKG 0.343150 0.031658      4


In [12]:
# ── Save year-level novelty time series ───────────────────────
yr_stats.to_csv(f'{OUT_DIR}/novelty_timeseries.csv', index=False)

# Also compute TGNN-based time series
tgnn_yr = (
    matrix[matrix['split'].isin(['SKG','NOVEL'])]
    .groupby(['year','split'])['tgnn_novelty']
    .agg(tgnn_mean='mean', tgnn_std='std')
    .reset_index()
)
tgnn_yr.to_csv(f'{OUT_DIR}/tgnn_timeseries.csv', index=False)
print("Saved novelty_timeseries.csv and tgnn_timeseries.csv")

Saved novelty_timeseries.csv and tgnn_timeseries.csv


## 5 — Emerging Concept Detection

Identify entities (research concepts) that show rapidly increasing connectivity in the knowledge graph over time — signals of emerging research trends.

In [13]:
# ── Entity degree per year (on SKG edges) ─────────────────────
# Use only Entity nodes (target side of knowledge edges)

entity_degree = defaultdict(lambda: defaultdict(int))

skg_tgt = skg_edges[['target','year']].copy()
for (ent, yr), cnt in skg_tgt.groupby(['target','year']).size().items():
    entity_degree[ent][yr] = cnt

# 2015–2020: chosen as the peak SKG growth period (see kg_snapshot_stats.csv);
# sufficient historical baseline (2010–2014) exists before this window.
ANALYSIS_YEARS = list(range(2015, 2021))

entity_records = []
for ent, yr_deg in entity_degree.items():
    vals = [yr_deg.get(y, 0) for y in ANALYSIS_YEARS]
    total = sum(vals)
    if total < 5:      # skip very rare entities
        continue
    # Compute growth: degree in last 2 years vs first 2 years
    early  = sum(vals[:2]) + 1e-9
    late   = sum(vals[-2:])
    growth = late / early
    entity_records.append({'entity': ent, 'total_degree': total, 'growth_ratio': growth,
                            **{str(y): v for y, v in zip(ANALYSIS_YEARS, vals)}})

entity_trend_df = pd.DataFrame(entity_records).sort_values('growth_ratio', ascending=False)

print(f"Entities analysed: {len(entity_trend_df):,}")
print("\nTop 20 Emerging Concepts (highest growth ratio):")
top_emerging = entity_trend_df.head(20)[['entity','total_degree','growth_ratio']]
print(top_emerging.to_string(index=False))

Entities analysed: 5,038

Top 20 Emerging Concepts (highest growth ratio):
      entity  total_degree  growth_ratio
E_3de0746a7d           246  2.450000e+11
E_5fe05153b7           165  1.520000e+11
E_4e00deceeb           160  1.220000e+11
E_8f26755056           114  1.040000e+11
E_d3052a2680            93  8.300000e+10
E_1d7c2923c1            97  6.800000e+10
E_10b3bbed74            91  5.500000e+10
E_a9784ab70a            90  5.300000e+10
E_79a2c5b09e            61  5.300000e+10
E_b61075c67b            57  4.900000e+10
E_3ab49a5de3            58  4.700000e+10
E_030bde5f90            60  4.400000e+10
E_7c899632ce            44  4.400000e+10
E_f54146a3fc            43  4.300000e+10
E_24d2a3cd80            45  4.300000e+10
E_45614f2250            53  4.200000e+10
E_bc92683377            41  4.000000e+10
E_276b2fd681            42  4.000000e+10
E_229e5b1363            40  4.000000e+10
E_8002477499            53  3.800000e+10


In [14]:
# ── Attach human-readable names from entity_nodes.csv ─────────
entity_nodes = pd.read_csv('../../outputs/final/entity_nodes.csv')
ent_name = dict(zip(entity_nodes['node_id'], entity_nodes['name']))

entity_trend_df['name'] = entity_trend_df['entity'].map(ent_name).fillna(entity_trend_df['entity'])
top_emerging_named = entity_trend_df[['entity','name','total_degree','growth_ratio']].head(30)

top_emerging_named.to_csv(f'{OUT_DIR}/emerging_concepts.csv', index=False)
print("Saved emerging_concepts.csv")
print(top_emerging_named[['name','growth_ratio']].head(20).to_string(index=False))

Saved emerging_concepts.csv
                                   name  growth_ratio
                                   bert  2.450000e+11
                            transformer  1.520000e+11
                            hidden size  1.220000e+11
                      transformer model  1.040000e+11
                        label smoothing  8.300000e+10
                                   adam  6.800000e+10
                              each word  5.500000e+10
                          hidden states  5.300000e+10
                          parallel data  5.300000e+10
                         length penalty  4.900000e+10
                   multi-head attention  4.700000e+10
                                seq2seq  4.400000e+10
               transformer architecture  4.400000e+10
                                   bart  4.300000e+10
                               6 layers  4.300000e+10
adam optimizer ( kingma and ba , 2014 )  4.200000e+10
  transformer ( vaswani et al. , 2017 )  4.000000e+10


## 6 — Incremental vs Disruptive Contribution Classification

Following the disruption index literature:  
- **Disruptive** papers: high novelty + low structural overlap with predecessors  
- **Incremental** papers: moderate novelty + high structural overlap

In [15]:
# ── Disruption index proxy ─────────────────────────────────────
# D = final_novelty_score – structural_novelty
# High D → disruptive (semantically novel beyond structural novelty)
# Low  D → incremental (structural novelty drives the score)

eval_mat = matrix[matrix['split'].isin(['SKG','NOVEL'])].copy()
eval_mat['disruption_index'] = eval_mat['final_novelty_score'] - eval_mat['structural_novelty']

# Classify using median threshold
d_med = eval_mat['disruption_index'].median()
eval_mat['contribution_type'] = np.where(
    eval_mat['disruption_index'] > d_med, 'Disruptive', 'Incremental'
)

print("Contribution type breakdown:")
print(eval_mat.groupby(['split','contribution_type']).size().unstack(fill_value=0))

print(f"\nMedian disruption index: {d_med:.4f}")
print(f"NOVEL papers classified disruptive: "
      f"{(eval_mat[eval_mat.split=='NOVEL'].contribution_type=='Disruptive').mean()*100:.1f}%")

Contribution type breakdown:
contribution_type  Disruptive  Incremental
split                                     
NOVEL                     559          203
SKG                      1280         1636

Median disruption index: -0.3783
NOVEL papers classified disruptive: 73.4%


In [16]:
# ── Save disruption classification ────────────────────────────
disrupt_out = eval_mat[['paper_id','year','split','domain',
                          'final_novelty_score','structural_novelty',
                          'disruption_index','contribution_type']].copy()
disrupt_out.to_csv(f'{OUT_DIR}/disruption_classification.csv', index=False)
print("Saved disruption_classification.csv")

# Top disruptive NOVEL papers (innovation case studies)
top_disruptive = (
    eval_mat[eval_mat['split']=='NOVEL']
    .nlargest(20, 'disruption_index')
    [['paper_id','title','domain','year','disruption_index','final_novelty_score']]
)
top_disruptive.to_csv(f'{OUT_DIR}/top20_disruptive_papers.csv', index=False)
print("\nTop 20 Disruptive NOVEL Papers:")
print(top_disruptive[['paper_id','domain','year','disruption_index']].to_string(index=False))

Saved disruption_classification.csv

Top 20 Disruptive NOVEL Papers:
             paper_id domain  year  disruption_index
 NOVEL_SA_W4295795036     SA  2022          0.657547
NOVEL_DIA_W1511604355    DIA  2025          0.604546
 NOVEL_MT_W4283721142     MT  2022          0.585869
 NOVEL_SA_W4380520708     SA  2022          0.565362
NOVEL_DIA_W4319439342    DIA  2023          0.544313
NOVEL_DIA_W4205178534    DIA  2022          0.504358
 NOVEL_MT_W4224050897     MT  2022          0.489549
 NOVEL_MT_W4205713819     MT  2022          0.489176
 NOVEL_MT_W4289261621     MT  2022          0.487649
 NOVEL_MT_W4210521984     MT  2022          0.483457
 NOVEL_MT_W4376871556     MT  2023          0.481601
 NOVEL_SA_W4295048810     SA  2022          0.481548
 NOVEL_MT_W4210920041     MT  2022          0.473098
 NOVEL_MT_W4385799520     MT  2023          0.465498
 NOVEL_MT_W4281901970     MT  2022          0.462379
 NOVEL_SA_W4220825176     SA  2022          0.458529
 NOVEL_MT_W4308165701     MT  

## 7 — Future Link Prediction (Emerging Innovation)

Predict entity pairs likely to become connected in the future by combining:
- **Common neighbours** — Jaccard similarity at year t
- **Temporal growth** — entity growth ratio from Section 5
- Interpret high-scoring pairs as potential future research directions

In [17]:
# ── Future link prediction on KG(2019) → predict 2020 ─────────
T_TRAIN  = 2019
T_FUTURE = 2020

kg_train  = skg_edges[skg_edges['year'] <= T_TRAIN]
kg_future = skg_edges[skg_edges['year'] == T_FUTURE]

# Build adjacency (entity → set of neighbours) at T_TRAIN using vectorised ops
nbr = defaultdict(set)
for src, tgt in zip(kg_train['source'], kg_train['target']):
    nbr[src].add(tgt)
    nbr[tgt].add(src)

def jaccard(a: str, b: str) -> float:
    na, nb = nbr[a], nbr[b]
    if not na or not nb:
        return 0.0
    return len(na & nb) / len(na | nb)

# Ground-truth future edges
future_pairs = set(zip(kg_future['source'], kg_future['target']))
existing_pairs = set(zip(kg_train['source'], kg_train['target']))
truly_new = future_pairs - existing_pairs

print(f"KG(train) edges : {len(existing_pairs):,}")
print(f"Future edges    : {len(future_pairs):,}")
print(f"Truly new links : {len(truly_new):,}")

# Sample candidate pairs for scoring
rng2 = np.random.default_rng(SEED)
all_nodes = list(nbr.keys())

# Positive: sample 500 truly-new links
truly_new_list = list(truly_new)
rng2.shuffle(truly_new_list)
pos_sample = truly_new_list[:500]

# Negative: sample 500 random non-existing pairs
neg_sample = []
while len(neg_sample) < 500:
    a, b = rng2.choice(all_nodes, 2, replace=False)
    if (a, b) not in future_pairs and (b, a) not in future_pairs:
        neg_sample.append((a, b))

scores = ([jaccard(a, b) for a, b in pos_sample] +
          [jaccard(a, b) for a, b in neg_sample])
labels = [1]*len(pos_sample) + [0]*len(neg_sample)

auc = roc_auc_score(labels, scores)
ap  = average_precision_score(labels, scores)
print(f"\nFuture Link Prediction (Jaccard)  AUC={auc:.4f}  AP={ap:.4f}")

KG(train) edges : 93,056
Future edges    : 35,015
Truly new links : 35,015

Future Link Prediction (Jaccard)  AUC=0.4980  AP=0.5000


In [18]:
# ── Top predicted future links (highest Jaccard at T_TRAIN) ───
# Sample candidate entity pairs from growing entities
growing_ents = entity_trend_df.head(50)['entity'].tolist()

future_candidates = []
for i, a in enumerate(growing_ents):
    for b in growing_ents[i+1:]:
        if (a,b) not in existing_pairs and (b,a) not in existing_pairs:
            score = jaccard(a, b)
            future_candidates.append({'entity_1': a, 'entity_2': b, 'jaccard_score': score})

future_cand_df = pd.DataFrame(future_candidates).sort_values('jaccard_score', ascending=False)
future_cand_df['entity_1_name'] = future_cand_df['entity_1'].map(ent_name).fillna(future_cand_df['entity_1'])
future_cand_df['entity_2_name'] = future_cand_df['entity_2'].map(ent_name).fillna(future_cand_df['entity_2'])

future_cand_df.head(30).to_csv(f'{OUT_DIR}/future_link_predictions.csv', index=False)
print("Saved future_link_predictions.csv")
print("\nTop predicted future links among emerging concepts:")
print(future_cand_df[['entity_1_name','entity_2_name','jaccard_score']].head(10).to_string(index=False))

Saved future_link_predictions.csv

Top predicted future links among emerging concepts:
       entity_1_name        entity_2_name  jaccard_score
                bart              roberta       1.000000
multi-head attention             6 layers       0.160000
            benefits     performance gain       0.150000
      length penalty multi-head attention       0.142857
                 gcn                nodes       0.125000
       parallel data   back - translation       0.113636
         transformer multi-head attention       0.113636
 multilingual models           finetuning       0.111111
    number of layers      number of heads       0.111111
         transformer      label smoothing       0.111111


## 8 — KG Snapshot Statistics & Evolution Summary

In [19]:
# ── Save KG snapshot growth data ──────────────────────────────
snapshot_df.to_csv(f'{OUT_DIR}/kg_snapshot_stats.csv', index=False)
print("Saved kg_snapshot_stats.csv")

# ── Final Summary ──────────────────────────────────────────────
print("\n" + "="*60)
print("TEMPORAL MODELING — FINAL SUMMARY")
print("="*60)
print(f"Temporal GNN  NOVEL vs SKG AUC : {best_auc:.4f}")
print(f"Link prediction mean AUC        : {lp_df['AUC'].mean():.4f}")
print(f"Future link prediction (Jaccard): AUC={auc:.4f}  AP={ap:.4f}")
print(f"Emerging concepts identified    : {len(top_emerging_named)}")
print(f"Disruption classifications      : {len(disrupt_out)}")
print("="*60)
print("\nOutput files written to outputs/final/:")
for f in ['tgnn_novelty_scores.csv','link_prediction_results.csv',
          'novelty_timeseries.csv','tgnn_timeseries.csv',
          'emerging_concepts.csv','disruption_classification.csv',
          'top20_disruptive_papers.csv','future_link_predictions.csv',
          'kg_snapshot_stats.csv']:
    print(f"  ✓ {f}")

Saved kg_snapshot_stats.csv

TEMPORAL MODELING — FINAL SUMMARY
Temporal GNN  NOVEL vs SKG AUC : 0.6661
Link prediction mean AUC        : 0.4867
Future link prediction (Jaccard): AUC=0.4980  AP=0.5000
Emerging concepts identified    : 30
Disruption classifications      : 3678

Output files written to outputs/final/:
  ✓ tgnn_novelty_scores.csv
  ✓ link_prediction_results.csv
  ✓ novelty_timeseries.csv
  ✓ tgnn_timeseries.csv
  ✓ emerging_concepts.csv
  ✓ disruption_classification.csv
  ✓ top20_disruptive_papers.csv
  ✓ future_link_predictions.csv
  ✓ kg_snapshot_stats.csv
